## Basic RAG

In [1]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters chromadb pypdf openai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/6

In [2]:
import os
import openai

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from chromadb.config import Settings

/tmp/ipykernel_1754/2720623074.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
import os
from google.colab import files

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    CSVLoader,
    UnstructuredWordDocumentLoader
)

# Upload file
uploaded = files.upload()

file_path = list(uploaded.keys())[0]

print("Uploaded File:", file_path)

# Detect extension
ext = os.path.splitext(file_path)[1].lower()

# Select loader
if ext == ".pdf":
    loader = PyPDFLoader(file_path)

elif ext == ".txt":
    loader = TextLoader(file_path)

elif ext == ".csv":
    loader = CSVLoader(file_path)

elif ext == ".docx":
    loader = UnstructuredWordDocumentLoader(file_path)

else:
    raise ValueError(f"Unsupported file type: {ext}")

# Load documents
documents = loader.load()

print("Documents Loaded:", len(documents))
print(documents[0].page_content[:1000])

Saving AI_Budget_Shopping_Assistant_RAG_Knowledge_Base.pdf to AI_Budget_Shopping_Assistant_RAG_Knowledge_Base.pdf
Uploaded File: AI_Budget_Shopping_Assistant_RAG_Knowledge_Base.pdf
Documents Loaded: 4
AI Budget Shopping Assistant
RAG Knowledge Base / Demo Product & Shopping Document
Purpose: This document is designed as a retrieval source for a Retrieval-Augmented Generation (RAG) system. It
contains synthetic product records, shopping policies, category guidance, and RAG implementation notes. The
product prices are for demonstration only and should be replaced or refreshed through a live product API in
production.
1. Product Catalog
ID
Product
Category
Price
Rating
Key Features
BP101
UrbanPack College Backpack
Backpack
■1499
4.3
25L; laptop compartment; water resistant; padded shoulder straps
BP102
TravelPro Laptop Backpack
Backpack
■1899
4.5
30L; 15.6-inch laptop sleeve; water resistant; USB charging port
BP103
CampusLite Backpack
Backpack
■999
4.1
20L; lightweight; multiple compartm

# Chunk Overlapping Example

Original Text
------------------------------------------------------------
Artificial intelligence is transforming healthcare by helping
doctors diagnose diseases earlier. Machine learning models
analyze medical images and patient records to detect patterns
that humans might miss. These tools improve accuracy...
------------------------------------------------------------

chunk size 11

### Without Overlap

Chunk 1: [Artificial intelligence ... diseases earlier.]

Chunk 2: [Machine learning models ... to detect]

Chunk 3: [patterns that humans ... diagnosis time.]

chunk overlap 4

### With Overlap (4 words)

Chunk 1: [Artificial intelligence ... doctors diagnose diseases earlier.]

                             ↓ overlap

Chunk 2: [doctors diagnose diseases earlier. Machine learning ... patient]

                                            ↓ overlap

Chunk 3: [analyze medical images and patient records to detect patterns...]

                                            ↓ overlap

Chunk 4: [detect patterns that humans might miss. These tools...]

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

print("Total Chunks:", len(docs))

Total Chunks: 21


In [9]:
from google.colab import userdata
api_key=userdata.get('api_key')
client = openai.OpenAI(
    api_key=api_key,
    base_url="https://nexusapi.navigatelabs.ai"
)

In [11]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name="rag",
    get_or_create = True

)

In [12]:
for i, doc in enumerate(docs):

    text = doc.page_content

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    embedding = response.data[0].embedding

    collection.add(
        documents=[text],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("Embeddings stored successfully!")

Embeddings stored successfully!


In [13]:
query = input("Ask your question: ")

Ask your question: give project overview


In [14]:
query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_embedding = query_response.data[0].embedding

In [15]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

In [16]:
retrieved_chunks = results["documents"][0]

for i, chunk in enumerate(retrieved_chunks):

    print(f"\nChunk {i+1}")
    print("-" * 50)

    print(chunk)


Chunk 1
--------------------------------------------------
AI Budget Shopping Assistant
RAG Knowledge Base / Demo Product & Shopping Document
Purpose: This document is designed as a retrieval source for a Retrieval-Augmented Generation (RAG) system. It
contains synthetic product records, shopping policies, category guidance, and RAG implementation notes. The
product prices are for demonstration only and should be replaced or refreshed through a live product API in
production.
1. Product Catalog
ID
Product
Category
Price
Rating
Key Features
BP101

Chunk 2
--------------------------------------------------
price
1499
Budget filtering / deterministic calculation
rating
4.3
Ranking signal
source_type
product_catalog
Traceability
document_type
product
Retrieval filtering
8. Recommended RAG Pipeline
User Query → Requirement Extraction → Query Embedding → Vector Search → Metadata Filtering → Product Data
Validation → Budget Combination Generation → Context Builder → LLM → Structured Recommen

In [17]:
context = "\n\n".join(retrieved_chunks)

In [18]:
prompt = f"""
Answer the question using the context below. If there is no context, you can answer on your own

Context:
{context}

Question:
{query}
"""

In [19]:
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response.choices[0].message.content

print("\nFINAL ANSWER")
print("=" * 50)

print(answer)


FINAL ANSWER
Based on the provided context, the project involves developing an AI-powered Budget Shopping Assistant that leverages Retrieval-Augmented Generation (RAG) techniques to help users find products within their budget constraints. The system utilizes a comprehensive product catalog containing synthetic product data, including categories, prices, ratings, and key features, along with shopping policies and category guidance.

Key components of the project include:

1. **Product Data Management:** Maintaining and retrieving relevant product information such as IDs, categories, prices, ratings, and features.
2. **RAG Retrieval Pipeline:** Implementing a structured process for user queries that involves requirement extraction, embedding generation, vector search, metadata filtering, product validation, and contextual building.
3. **Filtering and Optimization:** Applying deterministic filtering based on categories and budget, along with product ranking signals to suggest the best o

PDF

↓

Chunking

↓

Embeddings

↓

Vector DB

↓

Similarity Search

↓

Retrieved Context

↓

GPT-4.1-nano

↓

Final Answer

In [20]:
while True:

    query = input("\nAsk Question (type exit to quit): ")

    if query.lower() == "exit":
        break

    # Query embedding
    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    query_embedding = query_response.data[0].embedding

    # Retrieve documents
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    retrieved_chunks = results["documents"][0]

    context = "\n\n".join(retrieved_chunks)

    # Prompt
    prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    {query}
    """

    # LLM response
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    print("\nAnswer:")
    print(answer)


Ask Question (type exit to quit): what are the constrains in my project?

Answer:
Based on the provided context, the constraints in your project include:

1. **Budget Constraint**: The total price of the recommended products must be less than or equal to the user’s specified budget. Price calculations should be deterministic, and prices used are synthetic demonstration prices that need to be refreshed from the product API in a production environment.

2. **Product Relevance and Satisfaction**: Recommendations should consider the user's required categories, preferences, rating (minimum of 4.3), and useful features to ensure product relevance.

3. **No Valid Bundle Handling**: If no combination of products satisfies all required categories within the budget, you should explain this constraint to the user and suggest alternatives such as increasing the budget, choosing lower-priced products, or removing non-essential items.

4. **Price Reliability**: In a production setting, prices must 